In [ ]:
import numpy as np
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve

p = "../data/processed/"
tr = np.load(p + "windows_train.npz")
va = np.load(p + "windows_val.npz")
te = np.load(p + "windows_test.npz")

X_train, y_train = tr["X"], tr["y"]
X_val, y_val, s_val = va["X"], va["y"], va["serial"]
X_test, y_test, s_test = te["X"], te["y"], te["serial"]

stats = np.load(p + "normalization_stats.npz")
cols = list(stats["feature_cols"])
zero_norm = -stats["mean"] / stats["std"]      # normalized value that a raw value of 0 maps to

failed_set = set(pd.read_csv(p + "fail_dates.csv")["serial_number"])

print(X_train.shape, X_val.shape, X_test.shape)
print("random-guess PR-AUC -> val:", round(float(y_val.mean()), 5), "| test:", round(float(y_test.mean()), 5))

In [ ]:
def window_metrics(y, score):
    return {"PR-AUC": round(average_precision_score(y, score), 4),
            "ROC-AUC": round(roc_auc_score(y, score), 4)}

def pick_threshold(y, score):
    prec, rec, thr = precision_recall_curve(y, score)
    f1 = 2 * prec[:-1] * rec[:-1] / np.maximum(prec[:-1] + rec[:-1], 1e-12)
    return thr[np.argmax(f1)]

def alert_counts(y, score, thr):
    pred = score >= thr
    return {"true alerts": int((pred & (y == 1)).sum()),
            "false alerts": int((pred & (y == 0)).sum()),
            "missed positives": int((~pred & (y == 1)).sum())}

def drive_metrics(y, score, serial, thr):
    df = pd.DataFrame({"s": serial, "y": y, "a": score >= thr})
    pos_drives = set(df.loc[df["y"] == 1, "s"])
    detected = set(df.loc[(df["y"] == 1) & df["a"], "s"])
    healthy = df[~df["s"].isin(failed_set)]
    fa = healthy.groupby("s")["a"].any()
    return {"failed drives evaluated": len(pos_drives),
            "detected in last 7 days": len(detected),
            "healthy drives": len(fa),
            "healthy with a false alarm": int(fa.sum())}

In [ ]:
idx = [cols.index(c) for c in ["smart_5_raw", "smart_187_raw", "smart_197_raw"]]

def rule_score(X):
    last = X[:, -1, :][:, idx]
    return (last > zero_norm[idx] + 1e-4).sum(axis=1).astype(float)

for name, (X, y, s) in {"val": (X_val, y_val, s_val), "test": (X_test, y_test, s_test)}.items():
    sc = rule_score(X)
    print(name, "| rule: alert if at least one warning attribute is non-zero")
    print("  windows:", window_metrics(y, sc))
    print("  alerts: ", alert_counts(y, sc, 1))
    print("  drives: ", drive_metrics(y, sc, s, 1))

In [ ]:
def summarize(X):
    last = X[:, -1, :]
    return np.concatenate([last, X.max(axis=1), X.mean(axis=1), last - X[:, 0, :]], axis=1)

rng = np.random.default_rng(42)
pos = np.where(y_train == 1)[0]
neg = rng.choice(np.where(y_train == 0)[0], size=len(pos) * 30, replace=False)
idx_tr = rng.permutation(np.concatenate([pos, neg]))      # all positives + 30 negatives per positive

gb = HistGradientBoostingClassifier(max_depth=4, learning_rate=0.05, max_iter=300, random_state=42)
gb.fit(summarize(X_train[idx_tr]), y_train[idx_tr])

sc_val = gb.predict_proba(summarize(X_val))[:, 1]
sc_test = gb.predict_proba(summarize(X_test))[:, 1]
thr = pick_threshold(y_val, sc_val)          # threshold chosen on VALIDATION only

print("threshold from validation:", round(float(thr), 4))
for name, (y, sc, s) in {"val": (y_val, sc_val, s_val), "test": (y_test, sc_test, s_test)}.items():
    print(name, "| gradient boosting")
    print("  windows:", window_metrics(y, sc))
    print("  alerts: ", alert_counts(y, sc, thr))
    print("  drives: ", drive_metrics(y, sc, s, thr))